In [12]:
from pathlib import Path

from capstone_data.preprocessing.rebuild_dataset import (
    rebuild_dataset,
)


DATA_DIR = Path(
    r"C:\Users\Sheehizzle\Code\Capstone\Capstone-DATA4381\data"
)

results = rebuild_dataset(
    data_dir=DATA_DIR,
    output_dir=DATA_DIR / "final",
)

print(results)

FileExistsError: Output directory already contains files: C:\Users\Sheehizzle\Code\Capstone\Capstone-DATA4381\data\final

In [33]:
FINAL_PATH = Path(
    r"C:\Users\Sheehizzle\Code\Capstone\Capstone-DATA4381\data\final"
)

print(FINAL_PATH.exists())

True


In [34]:
from pathlib import Path
import numpy as np

from capstone_data.image_loader import (
    CLASS_NAMES,
    CLASS_TO_INDEX,
    IMAGE_EXTENSIONS,
    load_arrays,
    split_dataset,
)


FINAL_PATH = Path(
    r"C:\Users\Sheehizzle\Code\Capstone\Capstone-DATA4381\data\final"
)

dataset = []

for class_name in CLASS_NAMES:

    class_dir = FINAL_PATH / class_name

    for path in class_dir.iterdir():

        if (
            path.is_file()
            and path.suffix.lower() in IMAGE_EXTENSIONS
        ):

            dataset.append(
                {
                    "path": path,
                    "label": CLASS_TO_INDEX[class_name],
                    "class_name": class_name,
                }
            )

print(f"Records: {len(dataset)}")

Records: 16817


In [35]:
X, y = load_arrays(dataset)

print("X:", X.shape, X.dtype)
print("y:", y.shape, y.dtype)

print("Pixel range:", X.min(), X.max())

Loading images nerds:   0%|          | 0/16817 [00:00<?, ?it/s]

X: (16817, 48, 48) float32
y: (16817,) int64
Pixel range: 0.0 1.0


In [36]:
(
    X_train,
    y_train,
    X_val,
    y_val,
    X_test,
    y_test,
) = split_dataset(
    X,
    y,
    train_ratio=0.70,
    val_ratio=0.0,
    test_ratio=0.30,
    seed=42,
)

In [37]:
print("Train:")
print("X:", X_train.shape)
print("y:", y_train.shape)

print("\nValidation:")
print("X:", X_val.shape)
print("y:", y_val.shape)

print("\nTest:")
print("X:", X_test.shape)
print("y:", y_test.shape)

Train:
X: (11770, 48, 48)
y: (11770,)

Validation:
X: (0, 48, 48)
y: (0,)

Test:
X: (5047, 48, 48)
y: (5047,)


In [38]:
for split_name, labels in (
    ("Train", y_train),
    ("Test", y_test),
):

    classes, counts = np.unique(
        labels,
        return_counts=True,
    )

    print(f"\n{split_name}")
    print("-" * 30)

    for class_label, count in zip(
        classes,
        counts,
    ):
        class_name = CLASS_NAMES[class_label]

        print(
            f"{class_name:8s}: {count}"
        )


Train
------------------------------
angry   : 2162
happy   : 4673
sad     : 871
neutral : 4064

Test
------------------------------
angry   : 927
happy   : 2004
sad     : 374
neutral : 1742


In [39]:
assert (
    len(X_train)
    + len(X_val)
    + len(X_test)
    == len(X)
)

assert len(X_train) == len(y_train)
assert len(X_val) == len(y_val)
assert len(X_test) == len(y_test)

assert X_train.shape[1:] == (48, 48)
assert X_test.shape[1:] == (48, 48)

print("\n70/30 split passed.")


70/30 split passed.


In [40]:
from capstone_nn.layers import Flatten

flatten = Flatten()

X_flat = flatten.forward(X_train)

print("Before:", X_train.shape)
print("After: ", X_flat.shape)

assert X_flat.shape == (
    len(X_train),
    2304,
)

print("Flatten forward passed.")

Before: (11770, 48, 48)
After:  (11770, 2304)
Flatten forward passed.


In [41]:
import numpy as np

fake_gradient = np.ones_like(X_flat)

restored_gradient = flatten.backward(
    fake_gradient
)

print(
    "Incoming gradient:",
    fake_gradient.shape,
)

print(
    "Restored gradient:",
    restored_gradient.shape,
)

assert restored_gradient.shape == X_train.shape

print("Flatten backward passed.")

Incoming gradient: (11770, 2304)
Restored gradient: (11770, 48, 48)
Flatten backward passed.


In [42]:
from capstone_nn.layers import Dense

dense = Dense(
    input_size=2304,
    output_size=128,
)

small_batch = X_flat[:5]

z = dense.forward(
    small_batch
)

print("Input: ", small_batch.shape)
print("Output:", z.shape)

assert z.shape == (5, 128)

print("Dense forward passed.")

Input:  (5, 2304)
Output: (5, 128)
Dense forward passed.


In [43]:
print(z[0, :10])

[ 0.79751657 -0.06131417 -0.62503745  0.24935235 -1.10291885  0.93722708
  0.21756547 -1.37158724 -0.79793077  0.48093343]


In [44]:
from capstone_nn.activations import ReLU

relu = ReLU()

activation = relu.forward(z)

print("Before ReLU:")
print(z[0, :10])

print("\nAfter ReLU:")
print(activation[0, :10])

print("\nShapes:")
print(z.shape)
print(activation.shape)

assert activation.shape == z.shape
assert np.all(activation >= 0)

print("\nReLU forward passed.")

Before ReLU:
[ 0.79751657 -0.06131417 -0.62503745  0.24935235 -1.10291885  0.93722708
  0.21756547 -1.37158724 -0.79793077  0.48093343]

After ReLU:
[0.79751657 0.         0.         0.24935235 0.         0.93722708
 0.21756547 0.         0.         0.48093343]

Shapes:
(5, 128)
(5, 128)

ReLU forward passed.


In [45]:
relu = ReLU()

x = np.array(
    [[-2.0, 3.0, -1.0, 5.0]],
    dtype=np.float32,
)

y = relu.forward(x)

grad_output = np.array(
    [[0.4, -0.7, 1.2, 0.3]],
    dtype=np.float32,
)

grad_input = relu.backward(
    grad_output
)

print("Input:")
print(x)

print("\nForward output:")
print(y)

print("\nIncoming gradient:")
print(grad_output)

print("\nOutgoing gradient:")
print(grad_input)

Input:
[[-2.  3. -1.  5.]]

Forward output:
[[0. 3. 0. 5.]]

Incoming gradient:
[[ 0.4 -0.7  1.2  0.3]]

Outgoing gradient:
[[ 0.  -0.7  0.   0.3]]


In [48]:
from capstone_nn.layers import Dense

output_layer = Dense(
    input_size=128,
    output_size=4,
)

logits = output_layer.forward(
    activation
)

In [49]:
print(
    "Hidden activations:",
    activation.shape,
)

print(
    "Logits:",
    logits.shape,
)

assert logits.shape == (
    5,
    4,
)

print("Output Dense passed.")

Hidden activations: (5, 128)
Logits: (5, 4)
Output Dense passed.


In [50]:
print(logits)

[[ 0.95904287 -0.15346926 -0.21884496 -0.30292177]
 [ 0.00852007 -0.11781078 -0.13929809 -0.11349242]
 [ 0.46061886 -0.16645033  0.21976101 -0.5416792 ]
 [-0.33614292 -0.75278359  0.11974567 -0.0776478 ]
 [-0.24440062 -1.0888826   0.12665912 -0.16035576]]


In [51]:
from capstone_nn.losses import softmax

probabilities = softmax(
    logits
)

print("Logits:")
print(logits[0])

print("\nProbabilities:")
print(probabilities[0])

print(
    "\nProbability sum:",
    probabilities[0].sum(),
)

Logits:
[ 0.95904287 -0.15346926 -0.21884496 -0.30292177]

Probabilities:
[0.52089903 0.17123623 0.16039963 0.14746511]

Probability sum: 1.0


In [52]:
assert probabilities.shape == logits.shape

assert np.allclose(
    probabilities.sum(axis=1),
    1.0,
)

print("Softmax passed.")

Softmax passed.


In [58]:
import importlib
import capstone_nn.losses

importlib.reload(
    capstone_nn.losses
)


<module 'capstone_nn.losses' from 'C:\\Users\\Sheehizzle\\Code\\Capstone\\Capstone-DATA4381\\capstone_nn\\losses.py'>

In [59]:
from capstone_nn.losses import (
    softmax,
    cross_entropy,
)

In [60]:
probabilities = softmax(
    logits
)

small_labels = y_train[:5]

loss = cross_entropy(
    probabilities,
    small_labels,
)

print("True labels:")
print(small_labels)

print("\nProbabilities:")
print(probabilities)

print("\nLoss:")
print(loss)

True labels:
[3 3 1 3 1]

Probabilities:
[[0.52089903 0.17123623 0.16039963 0.14746511]
 [0.27555432 0.24285242 0.23768984 0.24390341]
 [0.37214241 0.19878173 0.29248657 0.13658929]
 [0.22066228 0.14547332 0.3481112  0.2857532 ]
 [0.25209699 0.10834623 0.36535651 0.27420027]]

Loss:
1.683148939472052


In [61]:
correct_probabilities = probabilities[
    np.arange(len(small_labels)),
    small_labels,
]

sample_losses = -np.log(
    correct_probabilities
)

print("Labels:")
print(small_labels)

print("\nCorrect-class probabilities:")
print(correct_probabilities)

print("\nIndividual losses:")
print(sample_losses)

print("\nMean loss:")
print(sample_losses.mean())

print("\nFunction loss:")
print(loss)

Labels:
[3 3 1 3 1]

Correct-class probabilities:
[0.14746511 0.24390341 0.19878173 0.2857532  0.10834623]

Individual losses:
[1.9141637  1.41098298 1.61554787 1.25262678 2.22242337]

Mean loss:
1.683148939472052

Function loss:
1.683148939472052
